In [ ]:
from dotenv import load_dotenv
from anthropic import Anthropic
import json
%pip install -U anthropic

load_dotenv()

client = Anthropic()
model ="claude-sonnet-4-5"

# Step 1: Draft a Prompt

In [ ]:
# Step 1: Prompt template, filled in per task later

def build_prompt_v1 (task):
    return f"""
Please provide a solution to the following task:

{task}
"""

# Step 2: Create an Eval Dataset

In [ ]:
# Step 2a: Helper Functions For using Claude to generate an Eval Dataset
def add_user_message(messages, text):
    user_message = ({"role": "user", "content": text})
    messages.append(user_message)

def add_assistant_message(messages, text):
    assitant_message = ({"role": "assistant", "content": text})
    messages.append(assitant_message)

def chat(messages, system = None, temperature = 1.0, stop_sequences=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "extra_body": {"temperature": temperature},
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return "".join(block.text for block in response.content if block.type == "text")


In [ ]:
# Step 2b + 3: Create custom dataset generation function and parse dataset:

def generate_dataset():
    prompt ="""
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.generate an evaluation dataset for a prompt evaluation.

Example output:
```json
[
    {
        "task": "Description of task",
        "format": "json" or "python" or "regex",
        "solution_criteria": "Key criteria for evaluating the solution"
    },
    ... additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages,"json")
    text = chat(messages, stop_sequences = ["```"])
    return json.loads(text)

# Test
dataset = generate_dataset()
print(dataset)

# Saving the Dataset

In [ ]:
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=2)